កំណត់បន្ថែមខាងក្រោមនេះត្រូវបានបង្កើតជាស្វ័យប្រវត្តិដោយ GitHub Copilot Chat ហើយមានចំណោមសម្រាប់ការតំឡើងដំបូងតែប៉ុណ្ណោះ។


# ការណែនាំអំពីវិស្វកម្មពន្យល់
វិស្វកម្មពន្យល់គឺជាដំណើរការរចនានិងបង្កើតពន្យល់សម្រាប់ភារកិច្ចដំណើរការ​ភាសានិយមធម្មជាតិ។ វា​រួម​បញ្ជូល​ការ​ជ្រើសរើស​ពន្យល់​ដែល​ត្រឹមត្រូវ ការ​កំណត់លក្ខណៈប៉ារ៉ាម៉ែត្រ​របស់​ពួកវា និង​ការ​វាយតម្លៃ​ប្រសិទ្ធភាព​របស់​ពួកវា។ វិស្វកម្មពន្យល់មានសារៈសំខាន់សម្រាប់សម្រេចបាននូវភាពត្រឹមត្រូវនិងប្រសិទ្ធភាពខ្ពស់ក្នុងគំរូ NLP។ នៅក្នុងផ្នែកនេះ ខ្ញុំនឹងស្រាវជ្រាវមូលដ្ឋាននៃវិស្វកម្មពន្យល់ដោយប្រើគំរូ OpenAI សម្រាប់ការស្រាវជ្រាវ។


### លំហាត់ ១៖ ការបែកសញ្ញា
ស្វែងយល់អំពីការបែកសញ្ញា ដោយប្រើប្រាស់ tiktoken ដែលជាឧបករណ៍បែកសញ្ញារហ័សប្រភពបើកពី OpenAI
មើល [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) សម្រាប់ឧទាហរណ៍បន្ថែម។


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]

### អនុញ្ញាត ២: ត្រួតពិនិត្យការតំឡើងកូនសោ OpenAI API

ដំណើរការកូដខាងក្រោមនេះ ដើម្បីផ្ទៀងផ្ទាត់ថា ចំណុចបញ្ចូល OpenAI របស់អ្នកត្រូវបានតំឡើងដោយបានត្រឹមត្រូវ។ កូដនេះព្យាយាមប្រើការផ្ដល់ព្រឹត្តិបត្រ ងាយៗមួយ ហើយធ្វើការត្រួតពិនិត្យនូវការបញ្ចប់។ អត្ថបទបញ្ចូល `oh say can you see` គួរត្រូវបញ្ចប់ជាប់ការសរសេរដូចជា `by the dawn's early light..`


In [ ]:
# The OpenAI SDK was updated on Nov 8, 2023 with new guidance for migration
# See: https://github.com/openai/openai-python/discussions/742

## Updated
import os
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2023-05-15"
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

## Updated
def get_completion(prompt):
    messages = [{"role": "user", "content": prompt}]       
    response = client.chat.completions.create(   
        model=deployment,                                         
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
        max_tokens=1024
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)

### លំហាត់ ៣៖ ការបង្កើតរឿងក្លែងបន្លំ  
ស្វែងយល់អំពីអ្វីដែលកើតឡើងពេលអ្នកសុំឲ្យ LLM ឆ្លើយតបនូវការបញ្ចប់សម្រាប់បញ្ហាដែលអាចជាបញ្ហាដែលមិនមាន ឬអំពីប្រធានបទដែលវាអាចមិនស្គាល់ ព្រោះវានៅក្រៅពីឈុតទិន្នន័យដែលបានបណ្តុះបណ្តាលរបស់វា (ថ្មីជាងនេះ)។ មើលថាពេលអ្នកសាកល្បងប្រើបញ្ហាផ្សេង ឬម៉ូដែលផ្សេង ការឆ្លើយតបប្រែប្រួលបែបណា។


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### កម្រងលំហាត់ ៤៖ ដាក់បន្ទាត់អនុវត្តិ
ប្រើអថេរ "text" ដើម្បីកំណត់មាតិកាចម្បង
ហើយប្រើអថេរ "prompt" ដើម្បីផ្ដល់ការណែនាំដែលពាក់ព័ន្ធនឹងមាតិកាចម្បងនោះ។

នៅទីនេះ យើងស្នើឲ្យម៉ូដែលសង្ខេបអត្ថបទសម្រាប់សិស្សថ្នាក់ទី២។


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### ការហាត់ប្រាណ ៥៖ សំណើបែបស្មុគស្មាញ  
សូមព្យាយាមសំណើដែលមានសាររបស់ប្រព័ន្ធ អ្នកប្រើ និងជំនួយការ  
ប្រព័ន្ធកំណត់បរិបទជំនួយការ  
សាររបស់អ្នកប្រើ និងជំនួយការផ្តល់បរិបទការសន្ទនាច្រើនជំហាន  

សូមចំណាំថា បុគ្គលិកលក្ខណៈជំនួយការត្រូវបានកំណត់ជា "ចម្លែក" ក្នុងបរិបទប្រព័ន្ធ។  
សូមព្យាយាមប្រើបរិបទបុគ្គលិកលក្ខណៈខុសគ្នា។ ឬសាកល្បងជួរចូល/ចេញផ្សេងទៀត។


In [ ]:
response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

### មហោស្រព៖ ស្វែងយល់អំពីអារម្មណ៍ក្នុងចិត្តរបស់អ្នក
ឧទាហរណ៍ខាងលើផ្តល់ឱ្យអ្នកនូវគំរូដែលអ្នកអាចប្រើដើម្បីបង្កើតពាក្យបញ្ជាថ្មីៗ (សាមញ្ញ, ស្មុគស្មាញ, ណែនាំ ល។) - សូមព្យាយាមបង្កើតមហោស្រពផ្សេងទៀតដើម្បីស្វែងយល់ពីគំនិតផ្សេងៗដែលយើងបានពិភាក្សាដូចជាឧទាហរណ៍, សញ្ញា និងផ្សេងៗទៀត។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**៖  
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលយើងខិតខំសម្រាប់ភាពត្រឹមត្រូវ សូមយល់ថាការបកប្រែម៉ាស៊ីនអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាតំបន់របស់វាគួរត្រូវបានគេចាត់ទុកជាអត្ថប្រយោជន៍ផ្លូវការជាទីបំផុត។ សម្រាប់ព័ត៌មានសំខាន់ ការបកប្រែ​មនុស្សវិជ្ជាជីវៈត្រូវបានណែនាំ។ យើងមិនអាចទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកប្រែមិនត្រឹមត្រូវណាមួយដែលកើតឡើងដោយសារការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
